<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Overfitting controls

**[Overfitting controls](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/)** · Deep Learning and Transformers Explained · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see overfitting in the validation loss when the accuracy hides it, control it with early stopping, dropout and weight decay, and decide when data augmentation keeps the label true.

| | |
|---|---|
| **Time** | About 35 minutes. The training runs take about 1 minute in total on the course's test computer. |
| **Needs** | An internet connection for the setup cells. No account. PyTorch (preinstalled in Colab and Kaggle). |
| **You should know** | Overfitting and regularization, from [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/) in the machine learning course. `model.eval()` and the best epoch, from [Batches, epochs and validation](https://learning.nextia-ai.com/courses/deep-learning/m03/batches-epochs-and-validation/). The curve shapes, from [Learning rate and optimizers](https://learning.nextia-ai.com/courses/deep-learning/m04/learning-rate-and-optimizers/). |
| **You do not need** | A graphics card or a local project. The setup below downloads the tickets and the course's code. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M04-L02-overfitting-controls/overfitting-controls-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: a plot helper

The next cell defines `plot_losses()`, which draws the training loss (dashed) and the validation loss (solid) of one or more runs. Run it. It prints nothing.

In [ ]:
# Helper for this lesson: plot the training and validation loss of one or more runs.
import matplotlib.pyplot as plt

def plot_losses(runs, title, top=None):
    """runs: {name: history}, each history a list of rows from train()."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for name, history in runs.items():
        epochs = [row["epoch"] for row in history]
        line, = ax.plot(epochs, [row["valid_loss"] for row in history], marker="o", ms=3, label=f"{name}: validation")
        ax.plot(epochs, [row["train_loss"] for row in history], ls="--", color=line.get_color(), label=f"{name}: training")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    if top is not None:
        ax.set_ylim(0, top)
    ax.set_title(title)
    ax.legend(fontsize=8)
    plt.show()

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The problem: a bigger router that gets worse while it trains

First, the bag-of-words tensors, as in Module 3. Run the cell.

In [ ]:
import torch
from torch import nn
from ticketnet import *       # the course's code: Vocab, bag_of_words, BagOfWordsNet, train() ...

train_rows, valid_rows = load_split("data/train.csv"), load_split("data/valid.csv")
vocab = Vocab([r["text"] for r in train_rows])          # words from the training tickets only
x_train, x_valid = bag_of_words(train_rows, vocab), bag_of_words(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)
print("x_train", tuple(x_train.shape), " x_valid", tuple(x_valid.shape))

> **Check.** You should see `x_train (4696, 2864)  x_valid (600, 2864)`.

Tomás gives the router 512 hidden neurons instead of 64 and trains it for 40 epochs with Adam's default learning rate (`"adamw"`, which is the same as Adam without weight decay).

> **Predict.** After 40 epochs, will the training accuracy be higher or lower than after 7? And the validation accuracy? Then run the cell. It takes about 10 seconds.

In [ ]:
set_seed(0)
big = BagOfWordsNet(len(vocab), hidden=512)            # 8 times more hidden neurons than before
print("parameters:", count_parameters(big))
history = train(big, x_train, y_train, x_valid, y_valid, epochs=40, optimizer="adamw", log=None)
for row in history:
    if row["epoch"] in (1, 2, 4, 7, 10, 20, 30, 40):
        print(f"epoch {row['epoch']:>2}: train loss {row['train_loss']:.4f}, accuracy {row['train_acc']:.2%}"
              f" | valid loss {row['valid_loss']:.4f}, accuracy {row['valid_acc']:.1%}")

> **Check.** You should see:
>
> `weights: 1469445`  
> `epoch  1: train loss 0.1588, accuracy 97.47% | valid loss 0.1437, accuracy 98.3%`  
> `epoch  2: train loss 0.0815, accuracy 98.51% | valid loss 0.0909, accuracy 99.2%`  
> `epoch  4: train loss 0.0293, accuracy 99.53% | valid loss 0.0730, accuracy 98.0%`  
> `epoch  7: train loss 0.0080, accuracy 99.96% | valid loss 0.0628, accuracy 97.8%`  
> `epoch 10: train loss 0.0034, accuracy 99.98% | valid loss 0.0714, accuracy 97.8%`  
> `epoch 20: train loss 0.0006, accuracy 100.00% | valid loss 0.0874, accuracy 97.5%`  
> `epoch 30: train loss 0.0002, accuracy 100.00% | valid loss 0.0971, accuracy 97.5%`  
> `epoch 40: train loss 0.0001, accuracy 100.00% | valid loss 0.1059, accuracy 97.3%`  

By epoch 20, the network gets every training ticket right, including the 70 tickets with a wrong label. That is memorising.

## 2. Accuracy hides it; the loss shows it

Run the cell. It draws the validation accuracy and the validation loss of the same run, side by side.

In [ ]:
import matplotlib.pyplot as plt
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.2))
epochs = [row["epoch"] for row in history]
left.plot(epochs, [100 * row["valid_acc"] for row in history], marker="o", ms=3)
left.set_ylim(90, 100); left.set_title("Validation accuracy (%)"); left.set_xlabel("Epoch")
right.plot(epochs, [row["valid_loss"] for row in history], marker="o", ms=3, color="tab:orange")
right.set_ylim(0, 0.2); right.set_title("Validation loss"); right.set_xlabel("Epoch")
plt.show()

> **Check.** On the left, the accuracy stays between 97.3% and 98.3% from epoch 4 on. On the right, the loss falls to 0.063 at epoch 7 and then rises to 0.106.

Why does the loss move while the accuracy does not? The next cell counts the wrong validation tickets at epoch 7 and at epoch 40, and the mean probability that the router gives its wrong answers. Run it.

In [ ]:
def wrong_and_sure(model):
    """How many validation tickets are wrong, and how sure the network is of those wrong answers."""
    model.eval()
    with torch.no_grad():
        probs = model(x_valid).softmax(1)
    wrong = probs.argmax(1) != y_valid
    return int(wrong.sum()), round(float(probs.max(1).values[wrong].mean()), 3)

set_seed(0)
at_epoch_7 = BagOfWordsNet(len(vocab), hidden=512)
train(at_epoch_7, x_train, y_train, x_valid, y_valid, epochs=7, optimizer="adamw", log=None)
print("epoch 7:  wrong tickets and mean probability of the wrong answer", wrong_and_sure(at_epoch_7))
print("epoch 40: wrong tickets and mean probability of the wrong answer", wrong_and_sure(big))

> **Check.** You should see:
>
> `epoch 7:  wrong tickets and mean probability of the wrong answer (13, 0.681)`  
> `epoch 40: wrong tickets and mean probability of the wrong answer (16, 0.816)`  

The overfitted router becomes sure of its mistakes. Watch the validation **loss** for overfitting.

## 3. Dropout on a tiny example

During training, dropout switches off each hidden neuron with a chance `p`, and scales the others up by 1 ÷ (1 − p). At evaluation, it does nothing. Run the cell.

In [ ]:
torch.manual_seed(3)
drop = nn.Dropout(p=0.5)                         # switch off each neuron with chance 0.5
h = torch.tensor([0.8, 0.2, 0.5, 0.6])           # the outputs of 4 hidden neurons
drop.train()                                     # training mode: dropout is on
print("training, pass 1:", [round(v, 2) for v in drop(h).tolist()])
print("training, pass 2:", [round(v, 2) for v in drop(h).tolist()])
drop.eval()                                      # evaluation mode: dropout is off
print("evaluation:      ", [round(v, 2) for v in drop(h).tolist()])

> **Check.** You should see:
>
> `training, pass 1: [1.6, 0.4, 0.0, 1.2]`  
> `training, pass 2: [0.0, 0.0, 1.0, 1.2]`  
> `evaluation:       [0.8, 0.2, 0.5, 0.6]`  

## 4. Compare the controls

Weight decay shrinks every weight a little at every step, so that only the weights that many tickets need stay large. The next cell trains the large router three times: no control, dropout 0.5, weight decay 0.05.

> **Predict.** Which control will give the lowest validation loss? Will any of them stop the rise after the best epoch? Then run the cell. It takes about 25 seconds.

In [ ]:
controls = {"no control": dict(dropout=0.0, weight_decay=0.0),
            "dropout 0.5": dict(dropout=0.5, weight_decay=0.0),
            "weight decay 0.05": dict(dropout=0.0, weight_decay=0.05)}
results = {}
for name, c in controls.items():
    set_seed(0)
    model = BagOfWordsNet(len(vocab), hidden=512, dropout=c["dropout"])
    results[name] = train(model, x_train, y_train, x_valid, y_valid, epochs=40, optimizer="adamw",
                          weight_decay=c["weight_decay"], log=None)
    best = min(results[name], key=lambda row: row["valid_loss"])
    last = results[name][-1]
    print(f"{name:<18} best epoch {best['epoch']:>2}: loss {best['valid_loss']:.3f}, accuracy {best['valid_acc']:.1%}"
          f" | epoch 40: loss {last['valid_loss']:.3f}, accuracy {last['valid_acc']:.1%}")

> **Check.** You should see:
>
> `no control         best epoch  7: loss 0.063, accuracy 97.8% | epoch 40: loss 0.106, accuracy 97.3%`  
> `dropout 0.5        best epoch  6: loss 0.056, accuracy 98.7% | epoch 40: loss 0.104, accuracy 97.0%`  
> `weight decay 0.05  best epoch  7: loss 0.063, accuracy 97.8% | epoch 40: loss 0.098, accuracy 97.2%`  

Run the next cell to see the three validation curves.

In [ ]:
plot_losses(results, "The large router with three controls", top=0.2)

Dropout gives the lowest point (0.056 at epoch 6). Weight decay 0.05 changes almost nothing. No control stops the rise: early stopping is what keeps the good router.

## 5. Early stopping with a patience

Early stopping keeps the weights of the epoch with the lowest validation loss. It stops when the loss has not improved for `patience` epochs.

> **Your turn.** Write `stop_epoch(history, patience)`. It walks through the epochs in order and returns two numbers: the best epoch so far when training stops, and the epoch after which training stops. If training never stops, return the best epoch and the last epoch. Then run the check cell.

In [ ]:
def stop_epoch(history, patience):
    best_loss, best_epoch, waited = float("inf"), None, 0
    for row in history:
        if row["valid_loss"] < best_loss:
            best_loss, best_epoch, waited = row["valid_loss"], row["epoch"], 0
        else:
            waited += 1
        if waited == patience:
            return best_epoch, row["epoch"]
    return best_epoch, history[-1]["epoch"]

print(stop_epoch(results["no control"], 5), stop_epoch(results["dropout 0.5"], 5))

In [ ]:
expect_hash('the result of stop_epoch()', [stop_epoch(results["no control"], 5), stop_epoch(results["dropout 0.5"], 5)], 'e6b339fe0bacbf24')

Now train the dropout router again for exactly its best number of epochs. `train()` starts from the same seed, so this run repeats the first one and stops at the best epoch. Run the cell.

In [ ]:
best = min(results["dropout 0.5"], key=lambda row: row["valid_loss"])
print("best epoch:", best["epoch"])
set_seed(0)
router = BagOfWordsNet(len(vocab), hidden=512, dropout=0.5)
train(router, x_train, y_train, x_valid, y_valid, epochs=best["epoch"], optimizer="adamw", log=None)
loss, acc = evaluate(router, x_valid, y_valid)
print(f"stopped at epoch {best['epoch']}: valid loss {loss:.3f}, accuracy {acc:.1%}")

> **Check.** You should see:
>
> `best epoch: 6`  
> `stopped at epoch 6: valid loss 0.056, accuracy 98.7%`  

## 6. Data augmentation: only where the label stays true

A shirt flipped from left to right is still a shirt (the case study [Classify clothing images](https://learning.nextia-ai.com/courses/deep-learning/m08/classify-clothing-images/) uses this). In text, one replaced word can change the team. Run the cell: it routes a delivery ticket and a copy where "parcel" became "refund".

In [ ]:
def route(text):
    router.eval()
    with torch.no_grad():
        probs = router(bag_of_words([{"text": text}], vocab)).softmax(1)[0]
    return TEAMS[int(probs.argmax())], round(float(probs.max()), 3)

for text in ["Where is my parcel? It is three days late.",
             "Where is my refund? It is three days late."]:
    print(route(text), text)

> **Check.** You should see:
>
> `('delivery', 0.934) Where is my parcel? It is three days late.`  
> `('delivery', 0.875) Where is my refund? It is three days late.`  

By Grace's policy, a missing refund goes to returns or payment, never to delivery. An augmented copy that keeps the label `delivery` would teach the router a wrong label, and the router already makes this mistake.

> **Your turn.** Try one replacement of your own with `route()`: change one word of a ticket so that the team changes by the policy (for example "arrived broken" against "broke after a month of use"). Write down what the router says and what the policy says. There is no check cell: the point is that you read the policy, not the router.

## 7. Change one thing: much stronger weight decay

> **Predict.** Weight decay 0.05 hardly changed anything. With 0.5 and with 5, does the best validation loss go down? Does the best epoch move? Then run the cell. It takes about 15 seconds.

In [ ]:
for weight_decay in [0.5, 5.0]:                    # 10 and 100 times stronger than 0.05
    set_seed(0)
    model = BagOfWordsNet(len(vocab), hidden=512)
    history = train(model, x_train, y_train, x_valid, y_valid, epochs=40, optimizer="adamw",
                    weight_decay=weight_decay, log=None)
    best, last = min(history, key=lambda row: row["valid_loss"]), history[-1]
    print(f"weight decay {weight_decay:<4} best epoch {best['epoch']:>2}: loss {best['valid_loss']:.3f},"
          f" accuracy {best['valid_acc']:.1%} | epoch 40: loss {last['valid_loss']:.3f},"
          f" train accuracy {last['train_acc']:.1%}")

> **Check.** You should see:
>
> `weight decay 0.5  best epoch  7: loss 0.065, accuracy 98.2% | epoch 40: loss 0.085, train accuracy 100.0%`  
> `weight decay 5.0  best epoch 29: loss 0.098, accuracy 98.8% | epoch 40: loss 0.108, train accuracy 99.4%`  

> **Your turn.** After the loop, `best` holds the best row of the last run (weight decay 5). Store its epoch in `wd5_best_epoch`, then run the check cell.

In [ ]:
wd5_best_epoch = best["epoch"]
print(wd5_best_epoch)

In [ ]:
expect('wd5_best_epoch', wd5_best_epoch, 29)

Strong weight decay stops the memorising (training accuracy 99.4%), but the best loss is much worse (0.098 against 0.063): the router is now less sure of its right answers too. It trades overfitting for underfitting.

## 8. Failure case: measuring with dropout still on

Tomás trains a small router with dropout 0.5 and checks one payment ticket three times, without `model.eval()`. Run the cell.

In [ ]:
set_seed(0)
small = BagOfWordsNet(len(vocab), dropout=0.5)       # 64 hidden neurons, dropout 0.5
train(small, x_train, y_train, x_valid, y_valid, epochs=8, log=None)
ticket = bag_of_words(valid_rows[:1], vocab)         # one payment ticket, shape (1, 2864)
small.train()                                        # Tomás's mistake: dropout is still on
with torch.no_grad():
    for run in range(3):
        print("training mode,   payment:", round(float(small(ticket).softmax(1)[0, 2]), 3))
small.eval()                                         # the right way to measure
with torch.no_grad():
    for run in range(3):
        print("evaluation mode, payment:", round(float(small(ticket).softmax(1)[0, 2]), 3))

> **Check.** You should see:
>
> `training mode,   payment: 0.996`  
> `training mode,   payment: 1.0`  
> `training mode,   payment: 0.983`  
> `evaluation mode, payment: 0.999`  
> `evaluation mode, payment: 0.999`  
> `evaluation mode, payment: 0.999`  

In training mode, dropout switches off a different half of the neurons in every pass, so the same ticket gets a different answer each time. Call `model.eval()` before you measure or use a network, and measure inside `torch.no_grad()`. The course's `evaluate()` does both.

## Recap

- A large network can memorise its training tickets; the validation loss shows it, even when the accuracy hardly moves.
- Early stopping keeps the best epoch; dropout and weight decay make memorising harder; none of them stops the rise alone.
- Augment only where the label stays true.
- Measure with `model.eval()`.

The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/) has the dropout diagram, the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Debug the experiment](https://learning.nextia-ai.com/courses/deep-learning/m04/debug-the-experiment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m04/overfitting-controls/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.